In this notebook I will load the behavioral data from the online Prolific cohort, as well as from the EMU cohort, and make figures for the paper:

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import arviz as az
import bambi as bmb
import argparse
from os.path import join
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import zscore
import os
import statsmodels.formula.api as smf
import statsmodels.api as sm


Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')

# Prolific Cohort

## Behavioral analysis of RPE on memory (delay condition)

In this project, we seek to understand how reward-prediction error information stacks upon perceptual memorability to modify memory performance. 

In this delay condition, participants perform two memory tests: one immediately after learning, and one 24 hours after.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from glob import glob
import sys 
sys.path.append('/hpc/users/qasims01/resources/ImgRatingSorter/')
from util import dprime
import seaborn as sns
import arviz as az

base_dir = '/sc/arion'
# save_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/'

sys.path.append('/hpc/users/qasims01/resources/utils/RLModels_K99/scripts')
import param_recov_utils
import model_recov_utils
from model_utils import *
from RWModel import RWModel
from WSLSModel import WSLSModel
from BayesModel import BayesModel
from RWModel_Fictive1 import RWModel_Fictive1
from RWModel_Fictive2 import RWModel_Fictive2
from RWModel_Signed import RWModel_Signed
from EWAModel import EWA
from VKFModel import VKFModel

from inspect import getmembers, isfunction
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
from sklearn.metrics import r2_score, roc_curve, auc

import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import chi2, ttest_ind, pearsonr, spearmanr, t, norm, zscore, wilcoxon, ranksums, mannwhitneyu, ttest_1samp
from scipy.ndimage import gaussian_filter


base_dir = '/sc/arion'
save_dir_plots = f'{base_dir}/projects/guLab/Salman/MemoryBanditManuscript/Figures/Exp2'
save_dir_data = f'{base_dir}/projects/guLab/Salman/MemoryBanditManuscript/Data/Exp2'

In [ ]:
# Set some plot defaults for matplotlib (when not using seaborn): 

import  matplotlib.font_manager

path = '/sc/arion/projects/guLab/Salman/Arial.ttf'
matplotlib.font_manager.fontManager.addfont(path)

prop = matplotlib.font_manager.FontProperties(fname=path)

# Set the default font family to the registered custom font
plt.rcParams['font.family'] = prop.get_name()

# Rest of your customization code
plt.rcParams['font.size'] = 20
plt.rcParams['lines.linewidth'] = 1

# Set palette: 

# Define RGB values for your two colors
ignacio_purple = np.array([86, 28, 70]) / 255
ignacio_blue = np.array([0, 70, 126]) / 255 

# Create a Seaborn palette with your custom colors normalized to 0-1
custom_palette = sns.color_palette([ignacio_purple, ignacio_blue])

ignacio_orange = np.array([255, 172, 77]) / 255
ignacio_mauve = np.array([193, 155, 211]) / 255
ignacio_beige = np.array([205, 182, 162]) / 255

# Set the custom palette
sns.set_palette(custom_palette)
sns.set_context("paper")


In [ ]:
# seaborn_colors = ['Accent', 'Accent_r', 'Blues', 'Blues_r', 'BrBG', 'BrBG_r', 'BuGn', 'BuGn_r', 'BuPu', 'BuPu_r', 
#  'CMRmap', 'CMRmap_r', 'Dark2', 'Dark2_r', 'GnBu', 'GnBu_r', 'Greens', 'Greens_r', 'Greys', 'Greys_r', 'OrRd', 
#  'OrRd_r', 'Oranges', 'Oranges_r', 'PRGn', 'PRGn_r', 'Paired', 'Paired_r', 'Pastel1', 
#  'Pastel1_r', 'Pastel2', 'Pastel2_r', 'PiYG', 'PiYG_r', 'PuBu', 'PuBuGn', 'PuBuGn_r', 
#  'PuBu_r', 'PuOr', 'PuOr_r', 'PuRd', 'PuRd_r', 'Purples', 'Purples_r', 'RdBu', 'RdBu_r', 
#  'RdGy', 'RdGy_r', 'RdPu', 'RdPu_r', 'RdYlBu', 'RdYlBu_r', 'RdYlGn', 'RdYlGn_r', 'Reds', 
#  'Reds_r', 'Set1', 'Set1_r', 'Set2', 'Set2_r', 'Set3', 'Set3_r', 'Spectral', 'Spectral_r', 
#  'Wistia', 'Wistia_r', 'YlGn', 'YlGnBu', 'YlGnBu_r', 'YlGn_r', 'YlOrBr', 'YlOrBr_r', 'YlOrRd', 
#  'YlOrRd_r', 'afmhot', 'afmhot_r', 'autumn', 'autumn_r', 'binary', 'binary_r', 'bone', 
#  'bone_r', 'brg', 'brg_r', 'bwr', 'bwr_r', 'cividis', 'cividis_r', 'cool', 'cool_r', 'coolwarm', 'coolwarm_r', 'copper', 'copper_r',
#  'cubehelix', 'cubehelix_r', 'flag', 'flag_r', 'gist_earth', 'gist_earth_r', 'gist_gray', 'gist_gray_r', 'gist_heat', 'gist_heat_r', 'gist_ncar', 'gist_ncar_r',
#  'gist_rainbow', 'gist_rainbow_r', 'gist_stern', 'gist_stern_r', 'gist_yarg', 
#  'gist_yarg_r', 'gnuplot', 'gnuplot2', 'gnuplot2_r', 'gnuplot_r', 'gray', 'gray_r',
#  'hot', 'hot_r', 'hsv', 'hsv_r', 'icefire', 'icefire_r', 'inferno', 
#  'inferno_r', 'magma', 'magma_r', 'mako', 'mako_r', 
#  'nipy_spectral', 'nipy_spectral_r', 'ocean', 'ocean_r', 'pink', 'pink_r',
#  'plasma', 'plasma_r', 'prism', 'prism_r', 'rainbow', 'rainbow_r',
#  'rocket', 'rocket_r', 'seismic', 'seismic_r', 'spring', 'spring_r',
#  'summer', 'summer_r', 'tab10', 'tab10_r', 'tab20', 'tab20_r', 'tab20b',
#  'tab20b_r', 'tab20c', 'tab20c_r', 'terrain', 'terrain_r', 'twilight',
#  'twilight_r', 'twilight_shifted', 'twilight_shifted_r', 'viridis', 'viridis_r', 'vlag', 'vlag_r', 'winter', 'winter_r']



In [ ]:
sns.set_context("paper")


In [ ]:
import time, sys
from IPython.display import clear_output

def update_progress(progress):
    bar_length = 20
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
    if progress < 0:
        progress = 0
    if progress >= 1:
        progress = 1

    block = int(round(bar_length * progress))

    clear_output(wait = True)
    text = "Progress: [{0}] {1:.1f}%".format( "#" * block + "-" * (bar_length - block), progress * 100)
    print(text)

In [ ]:
def compare_corr(xy, ab, n, n2 = None, twotailed=True):
    """
    Calculates the statistic significance between two independent correlation coefficients
    @param xy: correlation coefficient between x and y
    @param xz: correlation coefficient between a and b
    @param n: number of elements in xy
    @param n2: number of elements in ab (if distinct from n)
    @param twotailed: whether to calculate a one or two tailed test, only works for 'fisher' method
    @param conf_level: confidence level, only works for 'zou' method
    @param method: defines the method uses, 'fisher' or 'zou'
    @return: z and p-val
    """

    xy_z = 0.5 * np.log((1 + xy)/(1 - xy))
    ab_z = 0.5 * np.log((1 + ab)/(1 - ab))
    if n2 is None:
        n2 = n

    se_diff_r = np.sqrt(1/(n - 3) + 1/(n2 - 3))
    diff = xy_z - ab_z
    z = abs(diff / se_diff_r)
    p = (1 - norm.cdf(z))
    if twotailed:
        p *= 2

    return z, p

In [ ]:
def matthews_corrcoef(tp, tn, fp, fn):
    """
    """
    
    numerator = (tp * tn) - (fp * fn)
    
    denominator = ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5
    
    if denominator == 0:
        return 0
    
    return numerator / denominator

In [ ]:
from jax import config

import numpyro

numpyro.set_host_device_count(4)

config.update("jax_enable_x64", False)

def make_formula(y=None, X=None, Intx=None, rand_effect=None, rand_slopes_group=None, rand_slopes_predictor=None, categorical=None, categorical_baseline=True, rand_intx=False):
    if rand_effect is not None:
        # mixed/hierarchial/multilevel modeling:
        if categorical_baseline: # this means we set a baseline for categorical effects 
            rand_term = [f'(1|{x})' for x in rand_effect]
            if rand_slopes_group is not None: # do we want a more complex mixed-effects model with random slopes as well as intercepts?
                rand_slope_terms = [f'(1+'+'+'.join(rand_slopes_predictor)+f'|{b})' for b in rand_slopes_group]
                # [f'({a}|{b})' for a in rand_slopes_predictor for b in rand_slopes_group]
                if not Intx:
                    formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)
                else:
                    if rand_intx:
                        rand_slope_terms_Intx = [f'(1+'+'+'.join(Intx)+f'|{b})' for b in rand_slopes_group]
                        formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)+'+'+'+'.join(Intx)+'+'+'+'.join(rand_slope_terms_Intx)
                    else:
                        formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)+'+'+'+'.join(Intx)
            else:
                if not Intx: 
                    formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)
                else:
                    formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)
        else: # this means all levels of categorical effect are judged absent or present 
             # format random effects properly 
            rand_term = [f'(0+1|{x})' for x in rand_effect]
            if rand_slopes_group is not None: 
                rand_slope_terms = [f'(0+'+'+'.join(rand_slopes_predictor)+f'|{b})' for b in rand_slopes_group]
                if not Intx:
                    formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)
                else:
                    rand_slope_terms_Intx = [f'(0+'+'+'.join(Intx)+f'|{b})' for b in rand_slopes_group]
                    formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)+'+'+'+'.join(Intx)+'+'+'+'.join(rand_slope_terms_Intx)
            else:
                if not Intx:
                    formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)
                else:
                    formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)   

    return formula
    
    
def run_model(df, formula=None, categorical=None,
              priors=None, cores=2, chains=2, tune=1500, draws=2000, target_accept=0.95, model_fam='bernoulli', 
              output_dir='/sc/arion/work/qasims01/MemoryBanditData/EMU/BayesModels', return_model=False, save_model_res = True, label=None):
    """
    Run the model: 
    
    Parameters
    ----------
    df: pandas dataframe 
        All the variables 
    y: str
        Name of the dependent variable
    X: list of strings 
        Names of the independent variables 
    Intx: list of strings
        Names of the interactions, formatted as 'a:b' 
    rand_effect: str
        Name of variable for random effects (i.e. 'subject') 
    rand_slopes: bool 
        Should we fit random slopes in addition to random intercepts? 
    priors: dict 
        Prior distribution for any independent variables (i.e. Intercept, random intercept) 
        Default to weakly informative priors (Gelman et al. 2008) 
    categorical: list of strings 
        Names of categorical variables going into model 
    
    Returns
    -------
    results: model
    
    """

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # if no label provided, make your own
    if not label: 
        label = (f"{y}" + "_{}"*len(X)).format(*X)
    
        
    # construct the model 
    model = bmb.Model(formula=formula, 
              data=df,
             family=model_fam,
             priors=priors,
             categorical=categorical)
    
    # fit the model 
    results=model.fit(cores=cores, 
                          chains=chains, 
                          tune=tune, 
                          draws=draws, 
                          inference_method='mcmc',
                          target_accept=target_accept,
                         idata_kwargs={"log_likelihood": True})
    # ,init="advi+adapt_diag"

    # save out the CSV with the results 
    summary = az.summary(results, hdi_prob=0.95)
    for var in summary.index.values:
        if var.endswith("]"):  # skip individual subject parameters
            continue
        if var.startswith("1"):  # skip random effects.
            continue
        summary.loc[var, "P>0"] = np.mean(results.posterior[var].values > 0)
    summary.to_csv(join(output_dir , f"{label}_summary.csv"))

    
    if save_model_res:
        # save out the model for model comparison (THIS WILL BE GB-sized, so really only needed for model comparison)
        az.to_netcdf(results, join(output_dir , f"{label}_model"))
    
    if return_model:
        return model, results 
    
    
def plot_res(fitted_model, X, ax=None, save_label=None): 
    """
    Code for plotting select beta weights in a publication friendly manner from saved csv
    """

    axes = az.plot_forest(fitted_model,
                           kind='ridgeplot',
                           var_names=[f'^(?!.*\|).*{x}.*$' for x in X],
                           filter_vars="regex",
                           colors='gray',
                           combined=True,
                           ridgeplot_overlap=2,
                           ridgeplot_truncate=False,
                           ridgeplot_alpha=0.4,
                           hdi_prob=0.95,
                           ax=ax)
    # 'forestplot'
    ax.vlines(0, ax.get_ylim()[0], ax.get_ylim()[1], color = 'black')
    # plt.show()
#     plt.savefig(save_label, dpi=300)
#     plt.close()

def print_latex_table(fitted_model, X):
    """
    Code for printing the pub-friendly table of model details from model csv table
    """
    summary = az.summary(fitted_model, hdi_prob=0.95)
    
    # print out the latex table for the cell output
    results_df = summary.loc[:,
                   ['mean', 'sd', 'hdi_2.5%', 'hdi_97.5%']
                  ]
    all_tests = [] 
    for x in X: 
        temp = results_df.filter(like=f'{x}', axis=0)
        all_tests.append(temp)

    results_df = pd.concat(all_tests)

    c_string = '|c'*results_df.shape[0] + '|'
    print(results_df.reset_index().to_latex(index=False, 
                                            column_format=c_string).replace("\\\n", "\\ \hline\n"))  
    
def plot_predictions(df, model, fitted_model, X, y='recalled',  
                    save_dir='/home1/salman.qasim/Salman_Project/FR_Emotion/BayesModels'):
    """
    Code for plotting the model predictions 
    """
    
    # sample from posterior-predictive distribution to make in-sample predictions 
    model.predict(fitted_model, kind="pps")
    y_posterior = fitted_model.posterior[f"{y}_mean"].stack(samples=("chain", "draw")).values
    # Select 25% of the values in the posterior, making sure we take values from both chains.
    y_posterior = recall_posterior[:, ::4]

    for x in X:
        save_file = join(save_dir, f'{x}_PredictRecallplot.pdf')
        with PdfPages(save_file) as pdf:
            f, ax = plt.subplots(1, 1, figsize=[3,3], dpi=300)
            for pps in y_posterior.T:
                sns.regplot(data=df, x=x, y=pps, ci=None, scatter=False, line_kws={'alpha':0.1}, color='k')
            sns.regplot(data=df, x=x, y=y_posterior.mean(axis=1), ci=None, scatter=False, color='r')
            pdf.savefig()
            plt.close()

In [ ]:
def balance(seq):
    """
    
    Compute data set class balance using Shannon entropy divided by log(k) for k classes 
    
    """
    from collections import Counter
    from numpy import log
    
    n = len(seq)
    classes = [(clas,float(count)) for clas,count in Counter(seq).items()]
    k = len(classes)
    
    H = -sum([ (count/n) * log((count/n)) for clas,count in classes]) #shannon entropy
    return H/log(k)

# Data pre-processing

This should only really need to be done once unless introducing more data

In [ ]:
full_data_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/full_data_df_HierarchicalRW.csv')
learn_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/learn_df_HierarchicalRW.csv')


# Initial pilot data path 
load_dir = f'{base_dir}/projects/guLab/Salman/Prolific/DelayedMemoryTest'

# Load the demographics data to get 
demo_file = f'{load_dir}/demographics.csv'
demographics_df = pd.read_csv(demo_file, index_col=False, header=0)
demographics_df.loc[demographics_df.Age == 'DATA_EXPIRED', 'Age'] = np.nan
# Some patients demos are shifted oddly
demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Age'] = demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Sex']
demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Sex'] = demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Ethnicity simplified']
demographics_df = demographics_df[['Participant id', 'Age', 'Sex']]

demographics_df.rename(columns={'Participant id': 'participant',
                               'Age': 'age', 
                               'Sex': 'sex'}, inplace=True)


In [ ]:
full_data_df = full_data_df.merge(demographics_df)

In [ ]:
learn_df = learn_df.merge(demographics_df)

Out of N=208, N=148 (~71%) did the learning task well. 

In [ ]:
full_data_df[full_data_df['condition']=='day_2'].participant.unique().shape

Out of N=208, 166 (~80%) did the delayed memory task.

In [ ]:
full_data_df[(full_data_df.good_dm_performance) & (full_data_df['condition']=='day_2')].participant.unique().shape

All of the subjects who did the learning task well also returned for the delayed memory task. 

In [ ]:
full_data_df = full_data_df[full_data_df.good_dm_performance]

In [ ]:
learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

full_data_df['gamble_rt_log'] = np.log(full_data_df.gamble_rt)
full_data_df['recog_rt_log'] = np.log(full_data_df.recog_rt)

full_data_df['repeated_presentation'] = 0
for subj in full_data_df.participant.unique():
    subj_repeated_index = full_data_df.loc[(full_data_df.participant==subj) & (full_data_df.condition=='day_2'), 'img_path'].isin(full_data_df.loc[(full_data_df.participant==subj) & (full_data_df.condition=='day_1'), 'img_path'])
    


Get rid of bad trials (rt):

In [ ]:
learn_df = learn_df[learn_df.gamble_rt>0.3]
learn_df = learn_df[learn_df.gamble_rt<8]

drop_trials = pd.concat([full_data_df[full_data_df.gamble_rt<0.3], 
                          full_data_df[full_data_df.gamble_rt>8], 
                          full_data_df[full_data_df.recog_rt<0.3], 
                          full_data_df[full_data_df.recog_rt>8]])

full_data_df.drop(index=drop_trials.index, inplace=True)

In [ ]:

# full_data_df['zrpe'] = np.nan
# full_data_df['zpm'] = np.nan
# for subj in full_data_df.participant.unique(): 
# #     full_data_df['rpe'] = full_data_df['rpe'].fillna(0)
#     full_data_df['zrpe'][full_data_df.participant==subj] = (full_data_df['rpe'][full_data_df.participant==subj] - np.nanmean(full_data_df['rpe'][full_data_df.participant==subj])) / (2 * np.nanstd(full_data_df['rpe'][full_data_df.participant==subj]))
#     full_data_df['zpm'][full_data_df.participant==subj] = (full_data_df['image_dprime'][full_data_df.participant==subj] - np.nanmean(full_data_df['image_dprime'][full_data_df.participant==subj])) / (2 * np.nanstd(full_data_df['image_dprime'][full_data_df.participant==subj]))



Plot the relationship between reward and rpe: 


In [ ]:
learn_df['zreward'] = learn_df.reward.apply(lambda x: -1 if x==0 else x)
f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3))
sns.violinplot(data=learn_df, x='zreward', y='rpe', inner="quart", 
               fill=False, ax=ax)
# sns.regplot(data=learn_df, x='zreward', y='rpe', scatter=False, ci=None, ax=ax)
sns.despine()

In [ ]:
subj_level_df_mean = full_data_df[full_data_df.good_dm_performance].groupby(['participant', 'condition']).mean(numeric_only=True).reset_index()
subj_level_df_mean['points'] = subj_level_df_mean.apply(lambda x: x.reward * 100 * 80, axis=1)
    
    

08/24/2024: Take a peek at the Day 2 data, taking into account the repeated presentations: 


In [ ]:
full_data_df['repeated_presentation'] = 0
for subj in full_data_df.participant.unique():
    subj_repeated_index = full_data_df.loc[(full_data_df.participant==subj) & (full_data_df.condition=='day_2'), 'img_path'].isin(full_data_df.loc[(full_data_df.participant==subj) & (full_data_df.condition=='day_1'), 'img_path'])
    

In [ ]:
full_data_df.loc[(full_data_df.participant==subj) & (full_data_df.condition=='day_2')]

Note to self (8/24/24): - It seems I repeated ALL presentations of my prolific cohort, like a fucking retard 
- In the EMU cohort I have a much nicer design where this is not the case. Great. 

In [ ]:
mixed_model_rt.summary()

In [ ]:
full_data_df
full_data_df.dropna(subset=['recog_rt_log', 'prior_rpe'], inplace=True)

mixed_model_rt = smf.mixedlm(formula='C(hits) ~ 1 + zrpe', 
                             data=full_data_df, 
                             re_formula= '1 + zrpe',
                             groups=full_data_df['participant']).fit()

In [ ]:
sns.regplot(data=full_data_df, x='prior_rpe', y='hits', logistic=True, scatter=False)
sns.regplot(data=full_data_df, x='zrpe', y='hits', logistic=True, scatter=False)

In [ ]:
save_dir_plots

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_Fig_RWParams_Prolific.pdf') as pdf: 
    ax = sns.jointplot(data = learn_df[learn_df.good_dm_performance].groupby('participant').mean(numeric_only=True), 
                       x='alpha', y='beta', height=8, ratio=2, kind='kde', shade=True, cmap='Spectral_r', 
                       fill=True, cbar=True, xlim=[0, 1.1], ylim=[-2, 12])

    plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
    # get the current positions of the joint ax and the ax for the marginal x
    pos_joint_ax = ax.ax_joint.get_position()
    pos_marg_x_ax = ax.ax_marg_x.get_position()
    # reposition the joint ax so it has the same width as the marginal x ax
    ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
    # reposition the colorbar using new x positions and y positions of the joint ax
    ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
    ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
    ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)
    pdf.savefig()



In [ ]:
with PdfPages(f'{save_dir_plots}/Fig1_LearningBehavior.pdf') as pdf:
    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(6,4))
    sns.lineplot(data=learn_df[learn_df['good_dm_performance']], x='learn_trials', y= 'male', color='orange', ax=ax, errorbar='se')
    sns.lineplot(data=learn_df[learn_df['good_dm_performance']], x='learn_trials', y= 'female', color='purple', ax=ax, errorbar='se')
    plt.vlines([16, 33, 48, 65], 0, 1, color = 'black')
    plt.hlines(MB1_data[(MB1_data.block==1) & (MB1_data.Gender==1)].mean().prob, 0, 16, color = 'orange')
    plt.hlines(MB1_data[(MB1_data.block==2) & (MB1_data.Gender==0)].mean().prob, 16, 33, color = 'purple')
    plt.hlines(MB1_data[(MB1_data.block==3) & (MB1_data.Gender==1)].mean().prob, 33, 48, color = 'orange')
    plt.hlines(MB1_data[(MB1_data.block==4) & (MB1_data.Gender==0)].mean().prob, 48, 65, color = 'purple')
    plt.hlines(MB1_data[(MB1_data.block==5) & (MB1_data.Gender==1)].mean().prob, 65, 80, color = 'orange')

    plt.ylabel('p(draw)')
    plt.title(f'subject performance')
    sns.despine()
    pdf.savefig()

In [ ]:
# Let's investigate this drop-off in Block 4:
MB1_data = pd.read_excel('/sc/arion/projects/guLab/Salman/Prolific/MB1_data.xlsx')

In [ ]:
MB1_data[(MB1_data.block==5) & (MB1_data.Gender==1)].mean().prob

In [ ]:
with PdfPages(f'{save_dir_plots}/Fig1_RewardProbabilities.pdf') as pdf:
    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(6,4))
    sns.barplot(data=MB1_data, x='block', hue='Gender', y = 'prob', errorbar='se', palette=['purple', 'orange'], ax=ax)
#     sns.stripplot(data=MB1_data, x='block', y='prob', hue='Gender', palette=['purple', 'orange'], ax=ax, size=2, alpha=0.5)

    plt.ylabel('p(reward)')
    plt.title('reward probabilities')
    sns.despine()
    pdf.savefig()

In [ ]:
save_dir_plots

In [ ]:
# 7/13/23: Is there any effect of trial on memory? 
with PdfPages(f'{save_dir_plots}/Fig1_MemoryBehavior.pdf') as pdf:

    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,2))
    sns.lineplot(data=full_data_df[full_data_df.condition=='day_1'], x='learn_trials', y='hits', errorbar='se', ax=ax, color='maroon')
    plt.vlines([16, 33, 48, 65], 0, 1, color = 'black')
    ax.set(ylabel='p(subsequent hit)', title='memory performance', ylim=[0,1])
    sns.despine()
    
    pdf.savefig() 

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_RewardHists.pdf') as pdf:    
    f, ax = plt.subplots(1,3, dpi=300, figsize=(6,2))
    sns.histplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], x='points', ax=ax[0], kde=True, color='gray', element='step', label='gambling')
    sns.histplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], x='dprime', ax=ax[1], kde=True, color='gray', element='step', label='memory')
    sns.regplot(data=subj_level_df_mean[subj_level_df_mean.condition=='day_1'], x='points', y='dprime', order=1, ax=ax[2], 
                scatter_kws={'s':6, 'alpha':0.5}, color='gray')

    ax[2].set_xlabel('reward (a.u.)')
    ax[2].set_ylabel('dprime')
    sns.despine()
    f.tight_layout()
    pdf.savefig()

Computational Modeling Results:



In [ ]:
# save_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/plots'
with PdfPages(f'{save_dir_plots}/Prolific_model_comparison.pdf') as pdf:
    
    # aggregate the data
    dfs = []
    for pid in learn_df[learn_df['good_dm_performance']].participant.unique():
        df_comp_loo = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{pid}_model_comp.csv')
        # re-rank the diverging models as tied for worst 
    #         df_comp_loo['rank'][df_comp_loo.divergences>0] = 4
    #         # re-rank non-diverging models ranked 4th as 3rd 
    #         df_comp_loo['rank'][(df_comp_loo.divergences==0) & (df_comp_loo['rank']==4)] = 3
        dfs.append(df_comp_loo)
    df_comp_loos = pd.concat(dfs)
    df_comp_loos.rename(columns={'Unnamed: 0':'model'}, inplace=True)

    df_comp_loos['elpd_waic_raw']= -2*df_comp_loos['elpd_waic'] # comptue the actual information crtierion

    f, ax = plt.subplots(1,1, dpi=300, figsize=(8,4))
    # sns.barplot(data=df_comp_loos[df_comp_loos.divergences==0], x='elpd_waic', y='model', orient='h', errorbar='se',
    #            order = ['Bayes', 'RWF1',  'EWA', 'RWS','RWModel', 'VKF', 'WSLS'], estimator='sum')
    sns.boxplot(data=df_comp_loos, y='elpd_waic_raw', x='model',  order = ['RWF1', 'Bayes',  'VKF', 'WSLS', 'RWS', 'RWModel', 'EWA'], showcaps=False,boxprops={'facecolor':'None'}, showfliers=False,whiskerprops={'linewidth':0})
    sns.stripplot(data=df_comp_loos, y='elpd_waic_raw', x='model', order = ['RWF1', 'Bayes',  'VKF', 'WSLS',  'RWS', 'RWModel', 'EWA'], color='k', size=2, alpha=0.5)
    # Get the handles and labels. For this example it'll be 2 tuples
    # of length 4 each.
    handles, labels = ax.get_legend_handles_labels()
    plt.xlabel('model')
    plt.ylabel("WAIC")
    sns.despine()
    pdf.savefig()
    

Parameter Recovery:

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_parameter_recovery.pdf') as pdf:

    for model_name in [x[0] for x in getmembers(param_recov_utils) if x[0].endswith('parallel')]: 
        try:
            param_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/{model_name}.csv')
        except:
            continue
        num_params = param_df.columns.shape[0]//2

        # Plot 
        f, ax = plt.subplots(1, 3, dpi=300, figsize=(12,4))

        for ix, param in enumerate(param_df.columns[:num_params].tolist()): 
            if num_params > 1:
                ax[ix].set_title(f'{model_name}')
                ax[ix].plot(param_df[f'sim_{param}'], param_df[f'{param}'], 'o', color='darkred', alpha=.7)
                if 'beta' in param: 
                    ax[ix].plot([0, 20], [0, 20], 'k--')
                else: 
                    ax[ix].plot([-0.2, 1.2], [-0.2, 1.2], 'k--')
                ax[ix].set_xlabel(f'true {param}')
                ax[ix].set_ylabel(f'fitted {param}')
                f.tight_layout()
            else: 
                ax[ix].set_title(f'{model_name}')
                ax[ix].plot(param_df[f'sim_{param}'], param_df[f'{param}'], 'o', color='darkred', alpha=.7)
                if 'beta' in param: 
                    ax[ix].plot([0, 20], [0, 20], 'k--')
                else: 
                    ax[ix].plot([-0.2, 1.2], [-0.2, 1.2], 'k--')
                ax[ix].set_xlabel(f'true {param}')
                ax[ix].set_ylabel(f'fitted {param}')
                f.tight_layout
        pdf.savefig()
        


In [ ]:
param_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/RWModel_Fictive1_param_recov_parallel.csv')


In [ ]:
pearsonr(param_df.alpha_pos, param_df.sim_alpha_pos)




In [ ]:
pearsonr(param_df.alpha, param_df.sim_alpha)


In [ ]:
pearsonr(param_df.beta, param_df.sim_beta)


We choose the Fictive Rescorla Wagner because parameter recovery for the learning-rates is so much better.

In [ ]:
# Prep a list to store results 
all_res = [] 

full_data_df['img_cat'] = full_data_df.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

full_data_df['trials_in_block'] = np.nan
for trial in full_data_df.learn_trials.unique():    
    if (trial > 16 ) & (trial < 34):
        full_data_df['trials_in_block'][full_data_df.learn_trials==trial] = full_data_df['learn_trials'][full_data_df.learn_trials==trial] - 16
    elif (trial > 33 ) & (trial < 49):
        full_data_df['trials_in_block'][full_data_df.learn_trials==trial] = full_data_df['learn_trials'][full_data_df.learn_trials==trial] - 33
    elif (trial > 48 ) & (trial < 66):
        full_data_df['trials_in_block'][full_data_df.learn_trials==trial] = full_data_df['learn_trials'][full_data_df.learn_trials==trial] - 48
    elif (trial > 65 ):
        full_data_df['trials_in_block'][full_data_df.learn_trials==trial] = full_data_df['learn_trials'][full_data_df.learn_trials==trial] - 65

# refactor as a proportion of all trials
# full_data_df['trials_in_block'] = full_data_df['trials_in_block']/80
    

model_df = full_data_df[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'image_dprime', 'dprime', 
                           'alpha', 'beta', 'age', 'sex', 'reward', 'participant', 'hit_rate', 'condition',
                         'gamble_rt_log', 'recog_rt_log', '_merge', 'img_cat', 'trials_in_block', 'learn_trials']].copy()

model_df.rename(columns={'image_dprime':'pm'}, inplace=True)

model_df = model_df[model_df.condition=='day_1']

model_df['zrpe'] = np.nan
model_df['zpm'] = np.nan
model_df['prior_zrpe'] = np.nan
model_df['zurpe'] = np.nan
# model_df['prior_zurpe'] = np.nan



# model_df['rpe'] = model_df['rpe'].fillna(0)
# model_df['urpe'] = model_df['rpe']**2
model_df['urpe'] = np.abs(model_df['rpe'])
model_df['rpe_resid'] = np.nan

# model_df['urpe'] = model_df['rpe'].fillna(0)
model_df['zreward'] = model_df.reward.apply(lambda x: -1 if x==0 else x)
model_df.sort_values(by=['participant', 'learn_trials'], inplace=True)

for subj in model_df.participant.unique(): 
    model_df['zrpe'][model_df.participant==subj] = (model_df['rpe'][model_df.participant==subj] - np.nanmean(model_df['rpe'][model_df.participant==subj])) / (2 * np.nanstd(model_df['rpe'][model_df.participant==subj]))
    model_df['zpm'][model_df.participant==subj] = (model_df['pm'][model_df.participant==subj] - np.nanmean(model_df['pm'][model_df.participant==subj])) / (2 * np.nanstd(model_df['pm'][model_df.participant==subj]))
    model_df['zurpe'][model_df.participant==subj] = (model_df['urpe'][model_df.participant==subj] - np.nanmean(model_df['urpe'][model_df.participant==subj])) / (2 * np.nanstd(model_df['urpe'][model_df.participant==subj]))
#     ZRPE for previous trial
    model_df['prior_zrpe'][model_df.participant==subj] = model_df['zrpe'][model_df.participant==subj].shift(1).values
    # model_df['prior_zurpe'][model_df.participant==subj] = model_df['zurpe'][model_df.participant==subj].shift(1)

    try:
        formula = 'zrpe ~ 1 + zreward'
        rel_col = ['zrpe', 'zreward']
        residual_model = smf.ols(formula=formula, data=model_df[model_df.participant==subj]).fit()
        rpe_residuals = residual_model.resid

        model_df['rpe_resid'][model_df.participant==subj] = rpe_residuals
    except ValueError:
        print(subj)

# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Subject level traits need to be filled across NaNs
model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['alpha', 'beta', 'age', 'trials_in_block']

model_df.dropna(subset=['age'], inplace=True)
model_df.age = model_df.age.astype(int)
model_df['memory_outcome'] = model_df['memory_outcome'].astype(int)
model_df['hits'] = model_df['hits'].astype(int)

# Zscore independent variables: 
for factor in independent_vars: 
    model_df[f'z{factor}'] = (model_df[factor] - np.nanmean(model_df[factor])) / (2 * np.nanstd(model_df[factor]))




In [ ]:
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'rpe_resid', 'participant', 'zage', 'sex', 'zalpha', 'zbeta']
model_df.dropna(subset=rel_col)

In [ ]:
# # sns.regplot(data=model_df[model_df.urpe>=0], x='urpe', y='memory_outcome', scatter=False)
sns.regplot(data=model_df[model_df.rpe<=0], x='rpe', y='memory_outcome', logistic=True)
sns.regplot(data=model_df[model_df.rpe>=0], x='rpe', y='memory_outcome', logistic=True)
# sns.regplot(data=model_df, x='rpe', y='memory_outcome', logistic=True, scatter=False)


In [ ]:
# Is it the case that high memory is driven almost entirely by rpe=0, which is, technically, more than rpe<0?

In [ ]:
plt.hist(model_df.zrpe)

In [ ]:
sns.regplot(data=model_df, x='zrpe', y='memory_outcome', scatter=False, logistic=True, ci=None)
sns.regplot(data=model_df, x='zpm', y='memory_outcome', scatter=False, logistic=True, ci=None)

In [ ]:
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'participant', 'zage', 'sex', 'zalpha', 'zbeta']
model_df[rel_col].dropna()

In [ ]:
# REWARD MODEL: 
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'zreward', 'zurpe', 'prior_zrpe', 'participant', 'zage', 'sex', 'zalpha', 'zbeta']
# 'zage', 'sex', 
# # Drop nan data
model_df = model_df.dropna(subset=rel_col)

# logistic regression:

formula = 'memory_outcome ~ 1 + zreward + zpm + zalpha + zbeta + zage + sex + (1 + zreward + zpm|participant)'

categorical = ['sex', 'participant']

label = f'prolific_behavioral_model_zreward'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.95, model_fam='bernoulli', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
# REWARD MODEL: 
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'zreward', 'rpe_resid', 'participant', 'zage', 'sex', 'zalpha', 'zbeta']
# 'zage', 'sex', 
# # Drop nan data
model_df = model_df.dropna(subset=rel_col)

# logistic regression:

formula = 'memory_outcome ~ 1 + rpe_resid + zpm + zalpha + zbeta + zage + sex + (1 + rpe_resid + zpm|participant)'

categorical = ['sex', 'participant']

label = f'prolific_behavioral_model_rpe_resid'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.95, model_fam='bernoulli', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'zreward', 'zurpe', 'prior_zrpe', 'participant', 'zage', 'sex', 'zalpha', 'zbeta']
# 'zage', 'sex', 
# # Drop nan data
model_df = model_df.dropna(subset=rel_col)

# logistic regression:

formula = 'memory_outcome ~ 1 + zrpe + zpm + zalpha + zbeta + zage + sex + (1 + zrpe + zpm|participant)'

categorical = ['sex', 'participant']

label = f'prolific_behavioral_model_zrpe'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.95, model_fam='bernoulli', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4), sharex=True)

results = az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/prolific_behavioral_model_rpe_resid_model')

with PdfPages(f'{save_dir_plots}/prolific_behavioral_model.pdf') as pdf:    
    ax.set(xlabel='Weight', ylabel='Parameter')
    # f.suptitle(f'{region}_{band}')
    sns.despine(fig=f, ax=ax, top=True, right=True, left=True, bottom=False)
    X = ['rpe_resid',
     'zpm', 'zalpha', 'zbeta', 'zage', 'sex']
    plot_res(results, X, ax=ax)
    ax.set(xlabel = 'model weight')
    f.tight_layout()
    pdf.savefig(f)

9/14/2024: 

Ok, so the RPEs stop predicting memory when you residualize out the reward information. But again, I'm not really making that argument in the paper (reward obviously matters very much). 

In [ ]:
results_df = az.summary(results)

In [ ]:
results_df[results_df.index=='zrpe']

In [ ]:
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'zurpe', 'prior_zrpe', 'participant', 'zage', 'sex', 'zalpha', 'zbeta']
# 'zage', 'sex', 
# # Drop nan data
model_df = model_df.dropna(subset=rel_col)

# logistic regression:

formula = 'memory_outcome ~ 1 + prior_zrpe + zpm + zalpha + zbeta + zage + sex + (1|participant)'

categorical = ['sex', 'participant']

label = f'prolific_behavioral_model_prior_zrpe'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='bernoulli', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4), sharex=True)

results = az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/prolific_behavioral_model_prior_zrpe_model')

with PdfPages(f'{save_dir_plots}/prolific_behavioral_model_prior_zrpe.pdf') as pdf:    
    ax.set(xlabel='Weight', ylabel='Parameter')
    # f.suptitle(f'{region}_{band}')
    sns.despine(fig=f, ax=ax, top=True, right=True, left=True, bottom=False)
    X = ['prior_zrpe',
     'zpm', 'zalpha', 'zbeta', 'zage', 'sex']
    plot_res(results, X, ax=ax)
    ax.set(xlabel = 'model weight')
    f.tight_layout()
    pdf.savefig(f)

In [ ]:
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'zurpe', 'prior_zrpe', 'participant', 'zage', 'sex', 'zalpha', 'zbeta']
# 'zage', 'sex', 
# # Drop nan data
model_df = model_df.dropna(subset=rel_col)

# logistic regression:

formula = 'memory_outcome ~ 1 + zurpe + zpm + zalpha + zbeta + zage + sex + (1|participant)'

categorical = ['sex', 'participant']

label = f'prolific_behavioral_model_zurpe'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='bernoulli', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4), sharex=True)

results = az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/prolific_behavioral_model_zurpe_model')

with PdfPages(f'{save_dir_plots}/prolific_behavioral_model_zurpe.pdf') as pdf:    
    ax.set(xlabel='Weight', ylabel='Parameter')
    # f.suptitle(f'{region}_{band}')
    sns.despine(fig=f, ax=ax, top=True, right=True, left=True, bottom=False)
    X = ['zurpe',
     'zpm', 'zalpha', 'zbeta', 'zage', 'sex']
    plot_res(results, X, ax=ax)
    ax.set(xlabel = 'model weight')
    f.tight_layout()
    pdf.savefig(f)

In [ ]:
del model_dict

In [ ]:
model_dict = {'zrpe': az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/prolific_behavioral_model_zrpe_model'),
             'zreward_with_rpe': az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/prolific_behavioral_model_zreward_with_rpe_model'),
             'zreward': az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/prolific_behavioral_model_zreward_model')}



In [ ]:
df_compare = az.compare(model_dict)

In [ ]:
az.plot_compare(df_compare)

9/14/2024: Pure reward does a better job at explaining memory variance than RPE does. 

That's ok. Nowhere do I say that RPE is better than reward at explaining memory. I say it's being tracked (by dopamine) as a way of using reward during decision-making. So I want to identify its unique contribution to memory, after accounting for pure reward information.



In [ ]:
# sns.regplot(data=model_df, x='zurpe', y='hits', scatter=False)
sns.regplot(data=model_df, x='zrpe', y='memory_outcome', scatter=False)
sns.regplot(data=model_df, x='prior_zrpe', y='memory_outcome', scatter=False)

Iterate through the dataframes again and make sure the data actually looks good... 

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

save_dir_plots = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots'


# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']

# Let's get rid of the patients who do worse than chance on the DM task, or have no hits in the memory task

subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# Load all the raw subject behavioral data: 

base_dir = '/sc/arion'
full_data_df = []
learn_df = []

# iterate through the subjects 
for ix, cell, in subj_df.iterrows(): 
    
    subj = cell.MSSMCode.strip(' ')
    
    if cell.LongVersion==1:
        two_ab_ntrials = 80
        all_mem_file =f'{base_dir}/projects/guLab/Salman/EMU/all_mem_data_long.xlsx' 
        all_mem_df = pd.read_excel(all_mem_file, engine='openpyxl')
        all_mem_df = all_mem_df[['img_path', 'DPRIME']]
        all_mem_df.img_path = all_mem_df.img_path.apply(lambda x: x[:-4]+'.png' )
    
    elif cell.LongVersion==0:
        
        two_ab_ntrials = 60
        all_mem_file =f'{base_dir}/projects/guLab/Salman/EMU/all_mem_data_short.xlsx' 
        all_mem_df = pd.read_excel(all_mem_file, engine='openpyxl')
        all_mem_df = all_mem_df[['img_path', 'DPRIME']]
        all_mem_df.img_path = all_mem_df.img_path.apply(lambda x: x[:-4]+'.png' )
        
    day1_df = pd.read_csv(glob(f'{base_dir}/projects/guLab/Salman/EMU/{subj}/behav/Day1/*.csv')[0], index_col=None, header=0)
    
    # trim down to relevant columns 
    relevant_cols = ['MB1_draw_key.keys', 'MB1_draw_key.started', 'MB1_draw_key.rt','img_path', 'prob', 'block', 'Gender',
           'reward', 'participant', 'MB1_face.started', 'MB1_ITI_cross.started']
    
    # For some reason the trials and the data are coded in alternating rows
    trials = (day1_df.iloc[2:(two_ab_ntrials*2)+2:2]['trials_2.thisN'].astype(int)+1).tolist()
    mb_df = day1_df.iloc[2:(two_ab_ntrials*2)+2:2][relevant_cols].reset_index(drop=True)
    
    mb_df['trials'] = trials
    mb_df.rename(columns={'MB1_draw_key.keys': 'gamble',
                 'MB1_draw_key.started': 'gamble_start',
                 'MB1_face.started': 'feedback_start',
                 'MB1_ITI_cross.started': 'baseline_start',
                 'MB1_draw_key.rt': 'gamble_rt',
                 'Gender': 'gamble_label'},
                 inplace=True)
    mb_df['baseline_start'] = day1_df.iloc[3:(two_ab_ntrials*2)+3:2]['MB1_ITI_cross.started'].values.tolist()
    
    # Code reward as 1 or 0
    mb_df.reward = mb_df.reward/100
    # Code gamble as 1 or 0 (0 is male, 1 is female)
    mb_df.gamble = mb_df.gamble.apply(lambda x: 0 if x==1.0 else 1)
    mb_df.gamble_label = mb_df.gamble_label.apply(lambda x: 'male' if x==1.0 else 'female')

    mb_df = mb_df.merge(all_mem_df, on='img_path', how='left', indicator=False)
    
    # Add a filter for people with bad performance
    mb_df['good_dm_performance'] = True
    rew = mb_df.reward.sum()
    if rew<=(two_ab_ntrials//2): # person was worse than chance
        mb_df.good_dm_performance=False
        
    # Now, process the memory data
    
    if cell.LongVersion==1:

        relevant_cols = ['img_path', 'prob', 'block', 'Gender',
               'reward', 'participant', 'MEM2_recall_key.keys', 'MEM2_recall_key.rt', 'MEM2_recall_key.stopped', 'MEM2_stimuli.started',
               'MEM2_conf_slider.rating', 'MEM2_conf_slider.rt','trials_3.thisN', 'MEM2_jitter.started', 'trialIm']
        rm_df_day1 = day1_df.dropna(subset=['trials_3.thisN'])[relevant_cols].reset_index(drop=True)
        
        rm_df_day1['img_path'] = day1_df.img_path.dropna()[-80:]
        rm_df_day1['prob'] = day1_df.prob.dropna()[-80:]
        rm_df_day1['block'] = day1_df.block.dropna()[-80:]
        rm_df_day1['Gender'] = day1_df.Gender.dropna()[-80:]
        rm_df_day1['reward'] = day1_df.reward.dropna()[-80:]
        rm_df_day1['participant'] = day1_df.participant.dropna()[-80:]
        rm_df_day1.rename(columns={'MEM2_recall_key.keys': 'recog_choice',
                     'MEM2_recall_key.rt': 'recog_rt',
                     'MEM2_conf_slider.rating': 'conf_choice',
                     'MEM2_conf_slider.rt': 'conf_rt',
                     'MEM2_recall_key.stopped': 'recog_time', 
                     'MEM2_jitter.started': 'baseline_start',
                     'MEM2_stimuli.started': 'cue_on',
                     'trials_3.thisN': 'trials',
                     'Gender': 'gamble_label'},
                     inplace=True) 
        
        rm_df_day1.gamble_label = rm_df_day1.gamble_label.apply(lambda x: 'male' if x==1.0 else 'female')
        # Code reward as 1 or 0
        rm_df_day1.reward = rm_df_day1.reward/100
        # Code recog_choice 
        rm_df_day1['recog_choice_bool'] = rm_df_day1.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
        rm_df_day1.recog_choice = rm_df_day1.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

        rm_df_day1.trials = rm_df_day1.trials.astype(int) + 1
        rm_df_day1['img_path'] = rm_df_day1['trialIm']

        rm_df_merge_day1 = rm_df_day1.merge(mb_df, on='img_path', how='left', indicator=True, suffixes=(None, '_y'))

        # compute the hit-rates and false-alarm rates 
        hits = rm_df_merge_day1[(rm_df_merge_day1._merge=='both') & (rm_df_merge_day1.recog_choice_bool==1)].shape[0]
        misses = rm_df_merge_day1[(rm_df_merge_day1._merge=='both') & (rm_df_merge_day1.recog_choice_bool==0)].shape[0]

        false_alarms = rm_df_merge_day1[(rm_df_merge_day1._merge=='left_only') & (rm_df_merge_day1.recog_choice_bool==1)].shape[0]
        correct_rejections = rm_df_merge_day1[(rm_df_merge_day1._merge=='left_only') & (rm_df_merge_day1.recog_choice_bool==0)].shape[0]

        hm = hits+misses
        fc = false_alarms+correct_rejections

        hrs = hits/hm
        fars = false_alarms/fc

        dp = dprime(hrs, fars, hm, fc)

        rm_df_merge_day1['hits'] = (rm_df_merge_day1._merge=='both') & (rm_df_merge_day1.recog_choice_bool==1)
        rm_df_merge_day1['misses'] = (rm_df_merge_day1._merge=='both') & (rm_df_merge_day1.recog_choice_bool==0)
        rm_df_merge_day1['false_alarms'] = (rm_df_merge_day1._merge=='left_only') & (rm_df_merge_day1.recog_choice_bool==1)
        rm_df_merge_day1['correct_rejections'] = (rm_df_merge_day1._merge=='left_only') & (rm_df_merge_day1.recog_choice_bool==0)

        rm_df_merge_day1['hit_rate'] = hrs
        rm_df_merge_day1['false_alarm_rate'] = fars
        rm_df_merge_day1['dprime'] = np.nan

        if dp != float("-inf"):
            rm_df_merge_day1['dprime'] = dp 
            
        rm_df_merge_day1['condition'] = 'day_1'

        # Merge in the image DPRIME 
        rm_df_merge_day1 = rm_df_merge_day1.merge(all_mem_df, on='img_path', how='left', indicator=False)
        rm_df_merge_day1.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 
        
        
        # Process Day 2 
        if cell.Day2 == 1:
            rm_df_day2 = pd.read_csv(glob(f'{base_dir}/projects/guLab/Salman/EMU/{subj}/behav/Day2/*.csv')[0], index_col=None, header=0)

            rm_df_day2.rename(columns={'trialIm': 'img_path'}, inplace=True) 
                        
            num_dual_presentations = len(set(list(set(mb_df.img_path.values).intersection(set(rm_df_day1.img_path.dropna().values)))).intersection(set(rm_df_day2.img_path.dropna().values)))
            print(f'I made a mistake. {num_dual_presentations} images are presented for recall TWICE!')
            rm_df_day2['repeated_presentation'] = 0
            rm_df_day2['repeated_presentation'][rm_df_day2['img_path'].isin(rm_df_day1['img_path'])] =  1

            relevant_cols = ['img_path', 'participant', 'key_resp.keys', 'key_resp.rt', 'key_resp.stopped', 'image.started',
                       'new_slider.rating', 'new_slider.rt','trials.thisN', 'polygon.started', 'repeated_presentation']
            rm_df_day2 = rm_df_day2.dropna(subset=['trials.thisN'])[relevant_cols].reset_index(drop=True)

            rm_df_day2.rename(columns={'key_resp.keys': 'recog_choice',
                         'key_resp.rt': 'recog_rt',
                         'polygon.started': 'baseline_start',
                         'new_slider.rating': 'conf_choice',
                         'key_resp.stopped': 'recog_time',
                          'image.started': 'cue_on',
                         'new_slider.rt': 'conf_rt',
                         'trials.thisN': 'trials'},
                         inplace=True) 

            # Code recog_choice 
            rm_df_day2['recog_choice_bool'] = rm_df_day2.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
            rm_df_day2.recog_choice = rm_df_day2.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

            rm_df_day2.trials = rm_df_day2.trials.astype(int) + 1

            rm_df_merge_day2 = rm_df_day2.merge(mb_df, on='img_path', how='left', indicator=True, suffixes=(None, '_y'))

            # compute the hit-rates and false-alarm rates 
            hits = rm_df_merge_day2[(rm_df_merge_day2._merge=='both') & (rm_df_merge_day2.recog_choice_bool==1)].shape[0]
            misses = rm_df_merge_day2[(rm_df_merge_day2._merge=='both') & (rm_df_merge_day2.recog_choice_bool==0)].shape[0]

            false_alarms = rm_df_merge_day2[(rm_df_merge_day2._merge=='left_only') & (rm_df_merge_day2.recog_choice_bool==1)].shape[0]
            correct_rejections = rm_df_merge_day2[(rm_df_merge_day2._merge=='left_only') & (rm_df_merge_day2.recog_choice_bool==0)].shape[0]

            hm = hits+misses
            fc = false_alarms+correct_rejections

            hrs = hits/hm
            fars = false_alarms/fc

            dp = dprime(hrs, fars, hm, fc)

            rm_df_merge_day2['hits'] = (rm_df_merge_day2._merge=='both') & (rm_df_merge_day2.recog_choice_bool==1)
            rm_df_merge_day2['misses'] = (rm_df_merge_day2._merge=='both') & (rm_df_merge_day2.recog_choice_bool==0)
            rm_df_merge_day2['false_alarms'] = (rm_df_merge_day2._merge=='left_only') & (rm_df_merge_day2.recog_choice_bool==1)
            rm_df_merge_day2['correct_rejections'] = (rm_df_merge_day2._merge=='left_only') & (rm_df_merge_day2.recog_choice_bool==0)

            rm_df_merge_day2['hit_rate'] = hrs
            rm_df_merge_day2['false_alarm_rate'] = fars
            rm_df_merge_day2['dprime'] = np.nan

            if dp != float("-inf"):
                rm_df_merge_day2['dprime'] = dp    

            # Merge in the image DPRIME 
            rm_df_merge_day2 = rm_df_merge_day2.merge(all_mem_df, on='img_path', how='left', indicator=False)
            rm_df_merge_day2.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 
            rm_df_merge_day2['condition'] = 'day_2'

            full_mem_df = pd.concat([rm_df_merge_day1, rm_df_merge_day2])
            
        else: 
            full_mem_df = rm_df_merge_day1
        
        mb_df.participant=subj
        full_mem_df.participant=subj
        
        learn_df.append(mb_df)
        full_data_df.append(full_mem_df)
        
        
#     ##### Second, process the MEM2 data: 
    elif cell.LongVersion==0:
        relevant_cols = ['img_path', 'prob', 'block', 'Gender',
               'reward', 'participant', 'MEM2_recall_key.keys', 'MEM2_recall_key.rt',
               'MEM2_conf_slider.response', 'MEM2_conf_slider.rt','MEM2_trials.thisTrialN']
        rm_df = df.dropna(subset=['MEM2_trials.thisTrialN'])[relevant_cols].reset_index(drop=True)
        rm_df.rename(columns={'MEM2_recall_key.keys': 'recog_choice',
                     'MEM2_recall_key.rt': 'recog_rt',
                     'MEM2_conf_slider.response': 'conf_choice',
                     'MEM2_conf_slider.rt': 'conf_rt',
                     'MEM2_trials.thisTrialN': 'trials',
                     'Gender': 'gamble_label'},
                     inplace=True)
        mb_df['length'] = 'short'        
        rm_df['length'] = 'short'
    

        rm_df.gamble_label = rm_df.gamble_label.apply(lambda x: 'male' if x==1.0 else 'female')
        # Code reward as 1 or 0
        rm_df.reward = rm_df.reward/100
        # Code recog_choice 
        rm_df['recog_choice_bool'] = rm_df.recog_choice.apply(lambda x: 1 if x==2.0 else 0)
        rm_df.recog_choice = rm_df.recog_choice.apply(lambda x: 'old' if x==2.0 else 'new')

        rm_df.trials = rm_df.trials.astype(int) + 1

        rm_df_merge = rm_df.merge(mb_df, on='img_path', how='left', indicator=True, suffixes=(None, '_y'))

        # compute the hit-rates and false-alarm rates 
        hits = rm_df_merge[(rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==1)].shape[0]
        misses = rm_df_merge[(rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==0)].shape[0]

        false_alarms = rm_df_merge[(rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==1)].shape[0]
        correct_rejections = rm_df_merge[(rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==0)].shape[0]

        hm = hits+misses
        fc = false_alarms+correct_rejections

        hrs = hits/hm
        fars = false_alarms/fc

        dp = dprime(hrs, fars, hm, fc)

        rm_df_merge['hits'] = (rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==1)
        rm_df_merge['misses'] = (rm_df_merge._merge=='both') & (rm_df_merge.recog_choice_bool==0)
        rm_df_merge['false_alarms'] = (rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==1)
        rm_df_merge['correct_rejections'] = (rm_df_merge._merge=='left_only') & (rm_df_merge.recog_choice_bool==0)

        rm_df_merge['hit_rate'] = hrs
        rm_df_merge['false_alarm_rate'] = fars
        rm_df_merge['dprime'] = np.nan

        if dp != float("-inf"):
            rm_df_merge['dprime'] = dp    

        # Merge in the image DPRIME 
        rm_df_merge = rm_df_merge.merge(all_mem_df, on='img_path', how='left', indicator=False)
        rm_df_merge.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 
        mb_df = mb_df.merge(all_mem_df, on='img_path', how='left', indicator=False)
        mb_df.rename(columns={'DPRIME': 'image_dprime'}, inplace=True) 

        mb_df.participant=subj
        rm_df_merge.participant=subj

        li_mb.append(mb_df)
        li_mem.append(rm_df_merge)


In [ ]:
learn_df = pd.concat(learn_df)
learn_df['male'] = 0
learn_df['female'] = 0
learn_df.male = learn_df.apply(lambda x: 1 if x.gamble==0 else 0, axis=1)
learn_df.female = learn_df.apply(lambda x: 1 if x.gamble==1 else 0, axis=1)

In [ ]:
full_data_df = pd.concat(full_data_df)

In [ ]:
combined_df = full_data_df.merge(learn_df, on=['img_path', 'participant'], how='left', suffixes=['_mem', '_gamble'])


In [ ]:
# Load the hierarchical model results: 

RWHierarchical_EMU = az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/EMU/HierarchicalRWModel')


In [ ]:
model_df = az.summary(RWHierarchical_EMU)

In [ ]:

alpha = model_df[model_df.index.str.startswith("alpha[")]['mean'].values
beta = model_df[model_df.index.str.startswith("beta[")]['mean'].values


In [ ]:
# # load behavioral data
# learn_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/learn_df.csv')
# combined_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/combined_df.csv')

# Filter just to the participants in this notebook
learn_df = learn_df[learn_df.participant.isin(subj_ids)]
combined_df = combined_df[combined_df.participant.isin(subj_ids)]


# I enter the day information in stupidly 
combined_df.condition = combined_df.condition.apply(lambda x: 'Day1' if x=='day_1' else 'Day2' if x=='day_2' else np.nan)
combined_df['recog_time'] = combined_df['recog_time'].apply(lambda x: float(x) if (~pd.isna(x)) & (x!='None') else x)
combined_df['baseline_start_mem'] = combined_df['baseline_start_mem'].apply(lambda x: float(x) if (~pd.isna(x)) & (x!='None') else x)


# Add Age and Sex to both dataframes 
learn_df['age'] = np.nan
learn_df['sex'] = np.nan
combined_df['sex'] = np.nan
combined_df['age'] = np.nan

# Add alpha and beta
learn_df['alpha'] = np.nan
learn_df['beta'] = np.nan
learn_df['rpe'] = np.nan

# Add alpha and beta
combined_df['alpha'] = np.nan
combined_df['beta'] = np.nan
combined_df['rpe'] = np.nan

for ix, participant in enumerate(learn_df.participant.unique()): 
    learn_df['age'][learn_df.participant==participant] = subj_df[subj_df.MSSMCode==participant].Age.values[0]
    learn_df['sex'][learn_df.participant==participant] = subj_df[subj_df.MSSMCode==participant].Sex.values[0]
    combined_df['age'][combined_df.participant==participant] = subj_df[subj_df.MSSMCode==participant].Age.values[0]
    combined_df['sex'][combined_df.participant==participant] = subj_df[subj_df.MSSMCode==participant].Sex.values[0]
        
    learn_df['alpha'][learn_df.participant==participant] = alpha[ix]
    learn_df['beta'][learn_df.participant==participant] = beta[ix]
    
    combined_df['alpha'][combined_df.participant==participant] = alpha[ix]
    combined_df['beta'][combined_df.participant==participant] = beta[ix]
    
    a = learn_df.loc[learn_df.participant==participant, 'gamble'].values
    r = learn_df.loc[learn_df.participant==participant, 'reward'].values
    n_trials = len(a)
    # initialise Q values for each of the two options
    Qs = np.zeros((n_trials, 2))
    rpe = np.zeros(n_trials)

    # Initialize Q table
    Q = np.array([0.5, 0.5])
    
    for t in range(n_trials):
        # Update Q table
        rpe[t] = r[t] - Q[int(a[t])]
        Q[int(a[t])] = Q[int(a[t])] + alpha[ix] * (rpe[t])
        
    learn_df.loc[learn_df.participant==participant, 'rpe'] = rpe
    
# add RPE to the combined df dataframe: 
for ix, row in learn_df.iterrows():
    combined_df.loc[(combined_df.img_path==row.img_path) & (combined_df.participant==row.participant), 'rpe'] = row.rpe
    
    

In [ ]:
combined_df.condition

In [ ]:
# f, ax = plt.subplots(1,1, dpi=300, figsize=(4,2))
with PdfPages(f'{save_dir_plots}/Fig_BanditBehavior.pdf') as pdf:
    
    f, ax = plt.subplots(1,1, dpi=300, figsize=(6,4))
    sns.lineplot(data=learn_df, x='trials', y= 'male', color='orange', ax=ax, errorbar='se')
    sns.lineplot(data=learn_df, x='trials', y= 'female', color='purple', ax=ax, errorbar='se')
    plt.vlines([16, 33, 48, 65], 0, 1, color = 'black')
    plt.ylabel('p(draw)')
    plt.title(f'subject performance')
    sns.despine()
    pdf.savefig()
    
    

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv') 

In [ ]:
learn_df = learn_df[learn_df.gamble_rt>0.3]
# learn_df = learn_df[learn_df.gamble_rt<8]

drop_trials = pd.concat([combined_df[combined_df.gamble_rt_gamble<0.3],
                          combined_df[combined_df.recog_rt<0.3]])

combined_df.drop(index=drop_trials.index, inplace=True)

In [ ]:
combined_df

8/24/2024: Taking a peek at the effects of RPE etc on day 2 and repeated presentations: 

In [ ]:

# Create the Presentation_Type variable
combined_df['Presentation_Type'] = np.where(combined_df['condition'] == 'Day1', 'Day1_New',
                                     np.where((combined_df['condition'] == 'Day2') & (combined_df['repeated_presentation'] == 0), 'Day2_New', 'Day2_Repeat'))


# Prep a list to store results 
all_res = [] 

# combined_df['img_cat'] = combined_df.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

combined_df['trials_in_block'] = np.nan
for trial in combined_df.trials_gamble.unique():    
    if (trial > 16 ) & (trial < 34):
        combined_df['trials_in_block'][combined_df.trials_gamble==trial] = combined_df['trials_gamble'][combined_df.trials_gamble==trial] - 16
    elif (trial > 33 ) & (trial < 49):
        combined_df['trials_in_block'][combined_df.trials_gamble==trial] = combined_df['trials_gamble'][combined_df.trials_gamble==trial] - 33
    elif (trial > 48 ) & (trial < 66):
        combined_df['trials_in_block'][combined_df.trials_gamble==trial] = combined_df['trials_gamble'][combined_df.trials_gamble==trial] - 48
    elif (trial > 65 ):
        combined_df['trials_in_block'][combined_df.trials_gamble==trial] = combined_df['trials_gamble'][combined_df.trials_gamble==trial] - 65

model_df = combined_df[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'DPRIME_y', 'dprime', 
                           'alpha', 'beta', 'age', 'sex', 'participant', 'hit_rate', 'Presentation_Type', 
                        '_merge', 'trials_in_block', ]].copy()

model_df.rename(columns={'DPRIME_y':'pm'}, inplace=True)

# model_df = model_df[model_df.condition=='Day1']

model_df['zrpe'] = np.nan
model_df['zpm'] = np.nan
for subj in model_df.participant.unique(): 
    model_df['zrpe'][model_df.participant==subj] = (model_df['rpe'][model_df.participant==subj] - np.nanmean(model_df['rpe'][model_df.participant==subj])) / (2 * np.nanstd(model_df['rpe'][model_df.participant==subj]))
    model_df['zpm'][model_df.participant==subj] = (model_df['pm'][model_df.participant==subj] - np.nanmean(model_df['pm'][model_df.participant==subj])) / (2 * np.nanstd(model_df['pm'][model_df.participant==subj]))

# model_df['zrpe'] = model_df['zrpe'].fillna(0)

# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Subject level traits need to be filled across NaNs
model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['alpha', 'beta', 'age', 'trials_in_block']

model_df.dropna(subset=['age'], inplace=True)
model_df.age = model_df.age.astype(int)
model_df['memory_outcome'] = model_df['memory_outcome'].astype(int)
model_df['hits'] = model_df['hits'].astype(int)

# Zscore independent variables: 
for factor in independent_vars: 
    model_df[f'z{factor}'] = (model_df[factor] - np.nanmean(model_df[factor])) / (2 * np.nanstd(model_df[factor]))




In [ ]:
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'participant', 'Presentation_Type']
# 'zage', 'sex', 
# # Drop nan data
model_df = model_df.dropna(subset=rel_col)

# logistic regression:rpe + pm + Presentation_Type + rpe:Presentation_Type
lvls = ['Day1_New', 'Day2_New', 'Day2_Repeat']

formula = 'memory_outcome ~ 1 + zpm + C(Presentation_Type, levels=lvls)*rpe + (1 + zpm + C(Presentation_Type, levels=lvls)*rpe|C(participant))'

categorical = ['participant', 'Presentation_Type']

label = f'prolific_behavioral_model_emu_Day2'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=6000, draws=6000, target_accept=0.9, model_fam='bernoulli', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
az.plot_trace(results)
plt.tight_layout()



SO it would seem:

1. Day2_New: compared to Day1_New, the positive RPEs either stop upregulating memory, or negative RPEs begin to (check this) 

2. Day2_Repeat: compared to Day1_New, the positive RPEs have an even stronger effect on memory

In [ ]:
save_dir_plots

In [ ]:
with PdfPages(f'{save_dir_plots}/Exp2_Fig_RWParams_EMU.pdf') as pdf: 
    ax = sns.jointplot(data = learn_df[learn_df.good_dm_performance].groupby('participant').mean(numeric_only=True), 
                       x='alpha', y='beta', height=8, ratio=2, kind='kde', shade=True, cmap='Spectral_r', 
                       fill=True, cbar=True, xlim=[0, 1.1], ylim=[-2, 12])

    plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
    # get the current positions of the joint ax and the ax for the marginal x
    pos_joint_ax = ax.ax_joint.get_position()
    pos_marg_x_ax = ax.ax_marg_x.get_position()
    # reposition the joint ax so it has the same width as the marginal x ax
    ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
    # reposition the colorbar using new x positions and y positions of the joint ax
    ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
    ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
    ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)
    pdf.savefig()


Prepare the model data

In [ ]:
# Prep a list to store results 
all_res = [] 

# combined_df['img_cat'] = combined_df.apply(lambda x: 'male' if x.male==1 else 'female', axis=1)

combined_df['trials_in_block'] = np.nan
for trial in combined_df.trials_gamble.unique():    
    if (trial > 16 ) & (trial < 34):
        combined_df['trials_in_block'][combined_df.trials_gamble==trial] = combined_df['trials_gamble'][combined_df.trials_gamble==trial] - 16
    elif (trial > 33 ) & (trial < 49):
        combined_df['trials_in_block'][combined_df.trials_gamble==trial] = combined_df['trials_gamble'][combined_df.trials_gamble==trial] - 33
    elif (trial > 48 ) & (trial < 66):
        combined_df['trials_in_block'][combined_df.trials_gamble==trial] = combined_df['trials_gamble'][combined_df.trials_gamble==trial] - 48
    elif (trial > 65 ):
        combined_df['trials_in_block'][combined_df.trials_gamble==trial] = combined_df['trials_gamble'][combined_df.trials_gamble==trial] - 65

# refactor as a proportion of all trials
# full_data_df['trials_in_block'] = full_data_df['trials_in_block']/80
    

model_df = combined_df[['hits', 'false_alarms', 'misses', 'correct_rejections', 'rpe', 'DPRIME_y', 'dprime', 
                           'alpha', 'beta', 'age', 'sex', 'participant', 'hit_rate', 'condition',
                        '_merge', 'trials_in_block', ]].copy()

model_df.rename(columns={'DPRIME_y':'pm'}, inplace=True)

model_df = model_df[model_df.condition=='Day1']

model_df['zrpe'] = np.nan
model_df['zpm'] = np.nan
for subj in model_df.participant.unique(): 
    # model_df['rpe'] = model_df['rpe'].fillna(0)
    model_df['zrpe'][model_df.participant==subj] = (model_df['rpe'][model_df.participant==subj] - np.nanmean(model_df['rpe'][model_df.participant==subj])) / (2 * np.nanstd(model_df['rpe'][model_df.participant==subj]))
    model_df['zpm'][model_df.participant==subj] = (model_df['pm'][model_df.participant==subj] - np.nanmean(model_df['pm'][model_df.participant==subj])) / (2 * np.nanstd(model_df['pm'][model_df.participant==subj]))

# model_df['zrpe'] = model_df['zrpe'].fillna(0)

# Make a new response variable, memory outcome 
model_df['memory_outcome'] = model_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')

print(f"Dataset has a balance of {balance(model_df['memory_outcome'])} between memory choices")

# CODE THE VARIABLES: 
model_df['memory_outcome'][model_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
model_df['memory_outcome'][model_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1


# Subject level traits need to be filled across NaNs
model_df['alpha'] = model_df.groupby('participant')['alpha'].ffill()
model_df['alpha'] = model_df.groupby('participant')['alpha'].bfill()
model_df['beta'] = model_df.groupby('participant')['beta'].ffill()
model_df['beta'] = model_df.groupby('participant')['beta'].bfill()
      
# Define the dependent variable and the independent variables
dependent_var = 'memory_outcome'
independent_vars = ['alpha', 'beta', 'age', 'trials_in_block']

model_df.dropna(subset=['age'], inplace=True)
model_df.age = model_df.age.astype(int)
model_df['memory_outcome'] = model_df['memory_outcome'].astype(int)
model_df['hits'] = model_df['hits'].astype(int)

# Zscore independent variables: 
for factor in independent_vars: 
    model_df[f'z{factor}'] = (model_df[factor] - np.nanmean(model_df[factor])) / (2 * np.nanstd(model_df[factor]))




In [ ]:
rel_col = ['memory_outcome', 'zrpe', 'zpm', 'participant']
# 'zage', 'sex', 
# # Drop nan data
model_df = model_df.dropna(subset=rel_col)

# logistic regression:

formula = 'memory_outcome ~ 1 + zrpe + zpm + (1 + zrpe + zpm|participant)'

categorical = ['participant']

label = f'prolific_behavioral_model_emu'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=6000, draws=6000, target_accept=0.9, model_fam='bernoulli', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4), sharex=True)
with PdfPages(f'{save_dir_plots}/prolific_behavioral_model_emu.pdf') as pdf:    
    ax.set(xlabel='Weight', ylabel='Parameter')
    # f.suptitle(f'{region}_{band}')
    sns.despine(fig=f, ax=ax, top=True, right=True, left=True, bottom=False)
    X = ['zrpe',
     'zpm']
    plot_res(results, X, ax=ax)
    ax.set(xlabel = 'model weight')
    f.tight_layout()
    pdf.savefig(f)

In [ ]:
learn_df.to_csv(f'{load_dir}/learn_df_RWH.csv', index=False)
combined_df.to_csv(f'{load_dir}/full_df_RWH.csv', index=False)

In [ ]:
# Compare the reward between healthy and emu patients: 
full_data_df = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/full_data_df_RWF1.csv')

subj_level_df_prolific = full_data_df[full_data_df.good_dm_performance].groupby(['participant', 'condition']).mean().reset_index()
subj_level_df_prolific['points'] = subj_level_df_prolific.apply(lambda x: x.reward * 100 * 80, axis=1)
    
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
learn_df_emu = pd.read_csv(f'{load_dir}/learn_df.csv')
subj_level_df_emu = learn_df_emu[learn_df_emu.good_dm_performance].groupby(['participant']).mean().reset_index()
subj_level_df_emu['points'] = subj_level_df_emu.apply(lambda x: x.reward * 100 * 80, axis=1)
    
    

In [ ]:
with PdfPages(f'{save_dir_plots}/compare_reward_prolific_emu.pdf') as pdf:    

    f, ax = plt.subplots(1,1, dpi=300, figsize=(3,3), sharex=True)
    sns.despine()
    sns.distplot(subj_level_df_emu.points)
    sns.distplot(subj_level_df_prolific.points, color='red')
    pdf.savefig()

In [ ]:
from scipy.stats import ttest_ind
ttest_ind(subj_level_df_emu.points, 
         subj_level_df_prolific.points)